# 01 — Keşifsel Veri Analizi (EDA)

**Veri Seti:** PhiUSIIL Phishing URL Dataset (UCI ML Repository)  
**Amaç:** Veri setinin yapısını, dağılımlarını ve özellikler arası ilişkileri anlamak.

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

DATA_PATH = '../data/PhiUSIIL_Phishing_URL_Dataset.csv'
FIG_DIR = '../results/figures/'
FIG_KWARGS = dict(dpi=150, bbox_inches='tight')

import os
os.makedirs(FIG_DIR, exist_ok=True)

sns.set_theme(style='whitegrid', palette='Set2')

## 1. Veri Yükleme ve Genel Bakış

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f'Boyut: {df.shape}')
df.head(3)

In [ ]:
print('Sütun tipleri:')
print(df.dtypes.value_counts())
print('\nEksik değer sayısı:')
print(df.isnull().sum().sum())

## 2. Sınıf Dağılımı

In [ ]:
label_counts = df['label'].value_counts()
label_pct = df['label'].value_counts(normalize=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].bar(['Legitimate (0)', 'Phishing (1)'], label_counts.values, color=['steelblue', 'tomato'])
axes[0].set_title('Sınıf Dağılımı (Sayı)')
axes[0].set_ylabel('Örnek Sayısı')

axes[1].pie(label_counts.values, labels=[f'Legitimate\n{label_pct[0]:.1f}%', f'Phishing\n{label_pct[1]:.1f}%'],
            colors=['steelblue', 'tomato'], autopct='%1.1f%%', startangle=90)
axes[1].set_title('Sınıf Dağılımı (%)')

plt.tight_layout()
fig.savefig(f'{FIG_DIR}class_distribution.png', **FIG_KWARGS)
plt.show()

print(label_counts)

## 3. Sayısal Özellik İstatistikleri

In [ ]:
IDENTIFIER_COLS = ['FILENAME', 'URL', 'Domain', 'Title']
df_clean = df.drop(columns=[c for c in IDENTIFIER_COLS if c in df.columns])
df_clean.describe().T.round(3)

## 4. Özellik Dağılımları — Sınıflara Göre (Top 12)

In [ ]:
numeric_cols = df_clean.select_dtypes(include='number').columns.tolist()
numeric_cols = [c for c in numeric_cols if c != 'label']

top12 = numeric_cols[:12]

fig, axes = plt.subplots(3, 4, figsize=(16, 10))
axes = axes.flatten()

for i, col in enumerate(top12):
    for label_val, color in [(0, 'steelblue'), (1, 'tomato')]:
        data = df_clean[df_clean['label'] == label_val][col]
        axes[i].hist(data, bins=40, alpha=0.6, color=color,
                     label='Legitimate' if label_val == 0 else 'Phishing', density=True)
    axes[i].set_title(col, fontsize=9)
    axes[i].legend(fontsize=7)

plt.suptitle('Özellik Dağılımları — Sınıflara Göre', y=1.01, fontsize=13)
plt.tight_layout()
fig.savefig(f'{FIG_DIR}feature_distributions.png', **FIG_KWARGS)
plt.show()

## 5. Korelasyon Heatmap

In [ ]:
numeric_df = df_clean.select_dtypes(include='number')
corr = numeric_df.corr()

fig, ax = plt.subplots(figsize=(18, 15))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap='RdBu_r', center=0,
            square=True, linewidths=0.3, ax=ax, cbar_kws={'shrink': 0.8})
ax.set_title('Özellik Korelasyon Matrisi', fontsize=14)
plt.tight_layout()
fig.savefig(f'{FIG_DIR}correlation_heatmap.png', **FIG_KWARGS)
plt.show()

## 6. Hedef ile En Yüksek Korelasyonlu Özellikler

In [ ]:
target_corr = corr['label'].drop('label').abs().sort_values(ascending=False)
print('Hedef değişken ile en yüksek korelasyonlu 15 özellik:')
print(target_corr.head(15))

fig, ax = plt.subplots(figsize=(8, 6))
target_corr.head(20).plot(kind='barh', ax=ax, color='steelblue')
ax.set_xlabel('|Korelasyon|')
ax.set_title('Hedef ile En Yüksek Korelasyonlu Top-20 Özellik')
ax.invert_yaxis()
plt.tight_layout()
fig.savefig(f'{FIG_DIR}target_correlation.png', **FIG_KWARGS)
plt.show()

## 7. TLD Dağılımı

In [ ]:
if 'TLD' in df.columns:
    tld_counts = df['TLD'].value_counts().head(20)
    fig, ax = plt.subplots(figsize=(10, 5))
    tld_counts.plot(kind='bar', ax=ax, color='steelblue')
    ax.set_title('En Yaygın 20 TLD')
    ax.set_xlabel('TLD')
    ax.set_ylabel('Sayı')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    fig.savefig(f'{FIG_DIR}tld_distribution.png', **FIG_KWARGS)
    plt.show()

## 8. Vajrobol'un 5 Özelliğinin İncelenmesi

In [ ]:
vajrobol_features = ['URLSimilarityIndex', 'LineOfCode', 'NoOfExternalRef', 'NoOfImage', 'NoOfSelfRef']
vajrobol_available = [f for f in vajrobol_features if f in df_clean.columns]

fig, axes = plt.subplots(1, len(vajrobol_available), figsize=(16, 4))
if len(vajrobol_available) == 1:
    axes = [axes]

for ax, feat in zip(axes, vajrobol_available):
    for label_val, color in [(0, 'steelblue'), (1, 'tomato')]:
        data = df_clean[df_clean['label'] == label_val][feat]
        ax.hist(data, bins=50, alpha=0.6, color=color, density=True,
                label='Legitimate' if label_val == 0 else 'Phishing')
    ax.set_title(feat, fontsize=9)
    ax.legend(fontsize=7)

plt.suptitle("Vajrobol'un 5 Özelliği — Sınıf Dağılımları", fontsize=12)
plt.tight_layout()
fig.savefig(f'{FIG_DIR}vajrobol_features.png', **FIG_KWARGS)
plt.show()

## Özet

- Veri seti dengeli dağılıma yakın (%57 Legitimate, %42.8 Phishing)
- Eksik değer yok
- Bazı özellikler sınıflar arasında belirgin ayrışma gösteriyor
- Vajrobol'un seçtiği 5 özellik sınıflar arasında güçlü ayrım sağlıyor
- Bir sonraki adım: Ön işleme (02_preprocessing.ipynb)